In [3]:
import pandas as pd

postings = pd.read_csv('data/raw/job_postings.csv')
skills = pd.read_csv('data/raw/job_skills.csv')
summary = pd.read_csv('data/raw/job_summary.csv')

print("=== job_postings ===")
print(postings.shape)
print(postings.columns.tolist())
print(postings.head(2))

print("\n=== job_skills ===")
print(skills.shape)
print(skills.columns.tolist())
print(skills.head(2))

print("\n=== job_summary ===")
print(summary.shape)
print(summary.columns.tolist())
print(summary.head(2))

=== job_postings ===
(12217, 15)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
                                            job_link  \
0  https://www.linkedin.com/jobs/view/senior-mach...   
1  https://www.linkedin.com/jobs/view/principal-s...   

             last_processed_time   last_status got_summary got_ner  \
0  2024-01-21 08:08:48.031964+00  Finished NER           t       t   
1  2024-01-20 04:02:12.331406+00  Finished NER           t       t   

  is_being_worked                                     job_title  \
0               f              Senior Machine Learning Engineer   
1               f  Principal Software Engineer, ML Accelerators   

             company       job_location  first_seen search_city  \
0  Jobs for Humanity      New Haven, CT  2024-01-14  East Haven   
1             Aurora  Sa

In [4]:
df = postings.merge(skills, on='job_link', how='left') \
    .merge(summary, on='job_link', how='left')

print(df.shape)
print(df.columns.tolist())
df.head(2)

(12217, 17)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_summary']


,job_link,last_processed_time,last_status,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills,job_summary
0,https://www.linkedin.com/jobs/view/senior-mach...,2024-01-21 08:08:48.031964+00,Finished NER,t,t,f,Senior Machine Learning Engineer,Jobs for Humanity,"New Haven, CT",2024-01-14,East Haven,United States,Agricultural-Research Engineer,Mid senior,Onsite,"Machine Learning, Programming, Python, Scala, ...",Company Description\nJobs for Humanity is part...
1,https://www.linkedin.com/jobs/view/principal-s...,2024-01-20 04:02:12.331406+00,Finished NER,t,t,f,"Principal Software Engineer, ML Accelerators",Aurora,"San Francisco, CA",2024-01-14,El Cerrito,United States,Set-Key Driver,Mid senior,Onsite,"C++, Python, PyTorch, TensorFlow, MXNet, CUDA,...",Who We Are\nAurora (Nasdaq: AUR) is delivering...


In [5]:
# Check missing values per column
print(df.isnull().sum())

# Check distribution of job titles
print(df['job_title'].value_counts().head(20))

# Preview raw job summary text
print(df.loc[0, 'job_summary'])

job_link               0
last_processed_time    0
last_status            0
got_summary            0
got_ner                0
is_being_worked        0
job_title              0
company                0
job_location           1
first_seen             0
search_city            0
search_country         0
search_position        0
job_level              0
job_type               0
job_skills             5
job_summary            0
dtype: int64
job_title
Senior Data Engineer                                        285
Senior Data Analyst                                         163
Data Engineer                                               149
Senior MLOps Engineer                                       138
Data Analyst                                                137
Data Scientist                                              128
Lead Data Engineer                                          123
Senior Data Scientist                                       119
Data Architect                          

In [6]:
# Drop rows with missing critical fields
df = df.dropna(subset=['job_location', 'job_skills']).reset_index(drop=True)

# Check for duplicate postings
print(f"Duplicate job_links: {df['job_link'].duplicated().sum()}")
df = df.drop_duplicates(subset=['job_link']).reset_index(drop=True)

print(df.shape)

Duplicate job_links: 0
(12211, 17)


In [7]:
import re

def clean_job_summary(text):
    if not isinstance(text, str):
        return ""
    # Remove common boilerplate sections
    boilerplate_patterns = [
        r"Show more\s*Show less",
        r"equal opportunity employer.*",
        r"reasonable accommodations?.*",
        r"Capital One (does not|will not).*",
        r"protected veteran status.*",
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.DOTALL)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['job_summary_clean'] = df['job_summary'].apply(clean_job_summary)

# Compare lengths before/after cleaning
print(df['job_summary'].str.len().describe())
print(df['job_summary_clean'].str.len().describe())

count    12211.000000
mean      4276.430923
std       2293.547237
min         21.000000
25%       2561.500000
50%       3992.000000
75%       5728.000000
max      19197.000000
Name: job_summary, dtype: float64
count    12211.000000
mean      3800.707641
std       2021.089203
min          0.000000
25%       2389.000000
50%       3548.000000
75%       5001.000000
max      19177.000000
Name: job_summary_clean, dtype: float64


In [8]:
# Find rows where cleaning wiped everything out
empty_after_clean = df[df['job_summary_clean'].str.len() == 0]
print(f"Number of rows now empty: {len(empty_after_clean)}")

if len(empty_after_clean) > 0:
    print(empty_after_clean[['job_title', 'company']].head())
    print("\n--- Original text of first empty case ---")
    print(empty_after_clean.iloc[0]['job_summary'])

Number of rows now empty: 15
                        job_title  \
1488  RESEARCH DATA SPECIALIST II   
1704     RESEARCH DATA ANALYST II   
2156     RESEARCH DATA ANALYST II   
3214  RESEARCH DATA SPECIALIST II   
3529          Senior Data Analyst   

                                                company  
1488           California Department of State Hospitals  
1704                                           Caltrans  
2156  California Department of Forestry and Fire Pro...  
3214           California Department of State Hospitals  
3529             California Public Utilities Commission  

--- Original text of first empty case ---
Equal Opportunity Employer
The State of California is an equal opportunity employer to all, regardless of age, ancestry, color, disability (mental and physical), exercising the right to family care and medical leave, gender, gender expression, gender identity, genetic information, marital status, medical condition, military or veteran status, national ori

In [9]:
import re

def clean_job_summary(text):
    if not isinstance(text, str):
        return ""

    # Only strip boilerplate if it appears in the last 15% of the text
    # (safer than blanket removal from first occurrence)
    boilerplate_markers = [
        "show more", "equal opportunity employer",
        "protected veteran status", "reasonable accommodations",
    ]

    cutoff_point = int(len(text) * 0.85)
    tail = text[cutoff_point:].lower()

    for marker in boilerplate_markers:
        idx = text.lower().find(marker, cutoff_point)
        if idx != -1:
            text = text[:idx]
            break

    text = re.sub(r"\s+", " ", text).strip()
    return text

df['job_summary_clean'] = df['job_summary'].apply(clean_job_summary)

print(df['job_summary_clean'].str.len().describe())

count    12211.000000
mean      4256.025469
std       2293.421298
min         21.000000
25%       2541.500000
50%       3972.000000
75%       5708.000000
max      19177.000000
Name: job_summary_clean, dtype: float64


In [10]:
# Flag postings that are mostly procedural/administrative (heuristic: very generic gov-application language)
admin_keywords = ['CalCareer', 'Examination/Employment Application', 'Statement of Qualifications']
df['is_admin_posting'] = df['job_summary'].str.contains('|'.join(admin_keywords), case=False, na=False)

print(f"Admin/procedural postings detected: {df['is_admin_posting'].sum()}")

# Drop them for the matching engine (not representative of real job content)
df_filtered = df[~df['is_admin_posting']].reset_index(drop=True)
print(df_filtered.shape)

Admin/procedural postings detected: 24
(12187, 19)
